In [ ]:
# Import libraries
using LinearAlgebra, Distributions, GaussianProcesses
using CSV, DataFrames, JLD2
using PyCall, PyPlot
using Random

np = pyimport("numpy")
include("../src/utils/scale_utils.jl"); using .ScaleUtils
include("../src/utils/gp_utils.jl");    using .GPUtils

# Single-source the RNG seed (reused in the generation cell below).
SEED = 9
rng = MersenneTwister(SEED)
BLAS.set_num_threads(1)
np.random.seed(SEED)
Random.seed!(SEED);

In [ ]:
# Pick ONE synthetic-trajectory error-model configuration. Mirrors AmeryFutureObservations.jl's FL_* flags:
#   error model {wiener, indep} × code noise {on, off}, on the pointwise per-year GP emulator.
# TAG suffixes the saved obs (.npy) + figure (.pdf) so configurations never overwrite each other.
FUTURE_OBS_CONFIGS = Dict(
    "wiener_codeon_pointwise"       => (err="wiener", code=true,  emul="peryear", suffix="_wiener_code"),         # B
    "independent_codeon_pointwise"  => (err="indep",  code=true,  emul="peryear", suffix="_indep_code"),          # A
    "independent_codeoff_pointwise" => (err="indep",  code=false, emul="peryear", suffix="_indep_nocode"),        # C
    "wiener_codeoff_pointwise"      => (err="wiener", code=false, emul="peryear", suffix="_wiener_nocode"),       # D
)
CONFIG = "wiener_codeoff_pointwise"          # <- pick ONE

cfg = FUTURE_OBS_CONFIGS[CONFIG]
ERR_MODEL, CODE_NOISE, EMULATOR, TAG = cfg.err, cfg.code, cfg.emul, cfg.suffix
@assert ERR_MODEL in ("wiener","indep")
@assert EMULATOR == "peryear"
println("CONFIG=$CONFIG  ERR_MODEL=$ERR_MODEL  CODE_NOISE=$CODE_NOISE  EMULATOR=$EMULATOR  file-suffix='$TAG'")

In [ ]:
# Load input parameters from the respective CSV file
X_raw = CSV.read("../inputs/data/Amery_Input_Parameters_Filtered.csv", DataFrame);

# 1) Grab all column‐names as Symbols
cols = Symbol.(names(X_raw))

# 2) Remove the index‐column symbol
cols = filter(c -> c != :Column1, cols)
println("Column names: ", cols)

# 3) Now call get_scaled_matrix on the remaining columns
X_scaled_t, X_scalers, X_mins, X_maxs = ScaleUtils.get_scaled_matrix(X_raw, cols);

X_scaled = X_scaled_t'

In [ ]:
# 1) Load future simulated observable from SSP ensemble loaded from their respective CSV file into a DataFrame
# Grounded Volume Change
y_grd_vol_raw = CSV.read("../inputs/data/Amery_Grd_Vol_Change_time_series_2300_ssp5.csv", DataFrame)

# 2) Convert each to a plain 119×285 Array
# 2.1. Not(1) drops the first column
# 2.2. Matrix(...) converts the DataFrame to a plain Matrix 
# 2.3. collect(...) turns the transpose matrix into a real Array
y_grd_vol_fut = collect(Matrix(select(y_grd_vol_raw, Not(1))));

# 3) Build the list of “representative” years and their column‐indices
gap = 15
years = [y for y in 2015:gap:2300 if y != 2015]
# Since column 1 corresponds to year 2017, column i ↦ year = 2016 + i.
# So to get year Y, use col = (Y – 2016). Equivalently: (Y - 2017) + 1.
col_idx = (years .- 2016)

# 4) Subset columns of the 119×285 matrix:
y_grd_vol_sub = y_grd_vol_fut[:, col_idx]

years_all  = years
y_grd_vol_all = hcat(y_grd_vol_sub)

# 5) Build column names for these subset columns:
colnames_grd_vol = Symbol.("y_grd_vol_" .* string.(years_all))
println(colnames_grd_vol)

# 6) Wrap matrix in a DataFrame, then scale **each column** to [0,1]:
df_y_grd_vol_all = DataFrame(y_grd_vol_all, colnames_grd_vol)

y_grd_vol_scaled, y_grd_vol_scalers, y_grd_vol_mins, y_grd_vol_maxs =
    ScaleUtils.get_scaled_matrix(df_y_grd_vol_all, colnames_grd_vol);

In [ ]:
grd_vol_path = "../inputs/emulators/cumulative_grounded_volume_change_gap_$(gap)_emulators.jld2"

# Load the canonical pre-trained emulator ONLY — never retrain (a fresh fit would yield
# different predictions, hence different trajectories, run-to-run). The load itself happens
# in the next cell; here we just fail loudly if the artifact is missing.
@assert isfile(grd_vol_path) "Canonical emulator not found: $grd_vol_path"

In [ ]:
# Load dictionaries from disk:
gp_grd_vol_dict = load_emulators(grd_vol_path)

# Inspect available years:
println("Emulated years: ", sort(collect(keys(gp_grd_vol_dict))))

In [ ]:
θ_posterior_expert = np.load("../inputs/params/present_day_2015_posterior_100_draws.npy")

post = DataFrame(vmThresh_post=θ_posterior_expert[:,1], fricExp_post=θ_posterior_expert[:,2], 
    mu_scale_post=θ_posterior_expert[:,3], stiff_scale_post=θ_posterior_expert[:,4], 
    gamma0_post=θ_posterior_expert[:,5], melt_flux_post=θ_posterior_expert[:,6]);

# Get the total number of rows in the DataFrame
total_rows = nrow(post)

In [ ]:
# Per-year observational-error SD for cumulative grounded volume: σ(year) = 5.720·√(year − 2016).
years = 2017:2300
sigma = 5.720 .* sqrt.(years .- 2016)
sig_grd_vol = DataFrame(year = collect(years), sigma = sigma)

In [ ]:
# 1) Extract the future years and count them.
years_selected = years_all[1:end]         # e.g. [2030, 2045, …, 2285]
n_fut = length(years_selected)           # number of future years

σ_grd_vol = [sig_grd_vol[sig_grd_vol.year .== y, :sigma][1] for y in years_selected]
println("Sigma values for selected years (via comprehension): ", σ_grd_vol)

# [WIENER] Temporally-correlated (discrete-Wiener / Brownian) observational-error
# covariance for the CUMULATIVE grounded-volume observations:
#     Σ_o[i,j] = σ_a² · min(τ_i, τ_j),   τ = year − 2016,   σ_a = 5.720 Gt/yr.
# Diagonal ≡ old per-year σ² (marginals unchanged); off-diagonals add the
# cross-year correlation the independent-draw model was missing.
σ_a = 5.720
τ_wiener = years_selected .- 2016
Σ_o = [σ_a^2 * min(τ_wiener[i], τ_wiener[j]) for i in 1:n_fut, j in 1:n_fut]
@assert maximum(abs.(diag(Σ_o) .- σ_grd_vol.^2)) < 1e-6

In [ ]:
# 1) Per-year GP arrays + scalers.
gps_grd_vol = [gp_grd_vol_dict[y] for y in years_selected]
Y_gv_mins = [y_grd_vol_mins[Symbol("y_grd_vol_$(y)")] for y in years_selected]
Y_gv_maxs = [y_grd_vol_maxs[Symbol("y_grd_vol_$(y)")] for y in years_selected]

# 2) Posterior draws (all 100) and output matrix.
n_draws = total_rows
Y_grd_vol_obs = zeros(n_draws, n_fut)

# Re-seed immediately before drawing so this cell is IDEMPOTENT and matches AmeryFutureObservations.jl.
rng = MersenneTwister(SEED)

for j in 1:n_draws
    x = eachrow(post)[j]
    # 3a) rescale the six inputs into [0,1]:
    p_vm     = (x.vmThresh_post    - X_mins[:vmThresh])   / (X_maxs[:vmThresh]    - X_mins[:vmThresh])
    p_fric   = (x.fricExp_post     - X_mins[:fricExp])    / (X_maxs[:fricExp]     - X_mins[:fricExp])
    p_mu     = (x.mu_scale_post    - X_mins[:mu_scale])   / (X_maxs[:mu_scale]    - X_mins[:mu_scale])
    p_stiff  = (x.stiff_scale_post - X_mins[:stiff_scale])/ (X_maxs[:stiff_scale] - X_mins[:stiff_scale])
    p_gamma0 = (x.gamma0_post      - X_mins[:gamma0])     / (X_maxs[:gamma0]      - X_mins[:gamma0])
    p_melt   = (x.melt_flux_post   - X_mins[:melt_flux])  / (X_maxs[:melt_flux]   - X_mins[:melt_flux])
    θ = [p_vm;; p_fric;; p_mu;; p_stiff;; p_gamma0;; p_melt]'    # 6×1

    # 3b) mean trajectory (raw Gt) + per-year GP code covariance (predict_f; mirrors AmeryFutureObservations.jl):
    preds       = predict_f.(gps_grd_vol, Ref(θ))
    μ_gv_raws   = only.([p[1] for p in preds])
    var_gv_raws = [p[2][1] for p in preds]
    μ_gv_un = μ_gv_raws          .* (Y_gv_maxs .- Y_gv_mins) .+ Y_gv_mins
    s_gv_un = sqrt.(var_gv_raws) .* (Y_gv_maxs .- Y_gv_mins)
    Σ_code  = Diagonal(s_gv_un.^2)

    # 3c) draw the whole trajectory as ONE Gaussian for the chosen error model (+ code cov if on):
    Σ_obs  = ERR_MODEL == "wiener" ? Σ_o : diagm(σ_grd_vol.^2)
    Σ_draw = CODE_NOISE ? Symmetric(Σ_obs .+ Σ_code) : Symmetric(Σ_obs)
    Y_grd_vol_obs[j, :] = rand(rng, MvNormal(μ_gv_un, Σ_draw))
end
println("Generated $(n_draws)×$(n_fut) synthetic obs  [EMULATOR=$EMULATOR ERR_MODEL=$ERR_MODEL CODE_NOISE=$CODE_NOISE]")

In [ ]:
plt = pyimport("matplotlib.pyplot")
pyimport("scienceplots")
plt.style.use(["default", "science", "no-latex"])
FONTSIZE = 14

In [ ]:
fig, ax = subplots(figsize=(8, 5.5))

selections = [2, 13, 19, 27, 40, 45, 58, 70, 78, 85, 94, 99, 100]
colors = PyPlot.cm.gist_rainbow.(range(0, 1, length=size(selections, 1)))
for (k, i) in enumerate(selections)
    ax.plot(2030:15:2300, Y_grd_vol_obs[i, :], marker="o", markersize=6, color=colors[k], ls="dashed",
        markerfacecolor=colors[k])
end

ax.set_xlabel("Analysis year", fontsize=FONTSIZE, labelpad=10)
ax.set_ylabel("Cumulative grounded mass change (Gt)", fontsize=FONTSIZE)
ax.ticklabel_format(style="sci", scilimits=(-2, 4), useMathText=true)
ax.tick_params(axis="both", which="major", labelsize=FONTSIZE*0.8)
ax.xaxis.offsetText.set_fontsize(FONTSIZE*0.8)
ax.yaxis.offsetText.set_fontsize(FONTSIZE*0.8)
ax.set_xticks([2050, 2100, 2150, 2200, 2250, 2300])
ax.set_xlim(2030, 2300)
ax.grid(true, alpha=0.5, zorder=1)

mkpath("../outputs/plots/future_obs")
PyPlot.savefig("../outputs/plots/future_obs/Synthetic_future_trajectories$(TAG).pdf", bbox_inches="tight", dpi=300)

In [ ]:
# 2×2 comparison of the four configurations' synthetic future observations — reads the canonical generated
# obs (intermediate_data/future_observations); internal quick-look figure, titled. (Ports the old
# src/future_obs/plot_future_obs.jl comparison into this notebook.)
comparison_configs = [("A: independent + code unc.",      "seed9_indep_code"),
                        ("B: temporal-corr. + code unc.",   "seed9_wiener_code"),                   
                        ("C: independent, no code unc.",    "seed9_indep_nocode"),
                        ("D: temporal-corr., no code unc.", "seed9_wiener_nocode")]
sel_traj = [2, 13, 19, 27, 40, 45, 58, 70, 78, 85, 94, 99, 100]
mkpath("../outputs/plots/future_obs")
figc, axsc = PyPlot.subplots(nrows=2, ncols=2, figsize=(15, 10))
for (a, (ttl, ser)) in enumerate(comparison_configs)
    ax = axsc[(a-1)÷2 + 1, (a-1)%2 + 1]
    Yc = JLD2.load("../intermediate_data/future_observations/$(ser)_future_observations.jld2", "Y_grd_vol_obs")
    cols = PyPlot.cm.gist_rainbow.(range(0, 1, length=length(sel_traj)))
    for (k, i) in enumerate(sel_traj)
        ax.plot(2030:15:2300, Yc[i, :], marker="o", markersize=4, ls="dashed", color=cols[k], markerfacecolor=cols[k])
    end
    ax.set_title(ttl, fontsize=FONTSIZE*0.9)
    ax.set_xticks([2050, 2100, 2150, 2200, 2250, 2300]); ax.set_xlim(2030, 2300)
    ax.ticklabel_format(style="sci", scilimits=(-2, 4), useMathText=true)
    ax.tick_params(axis="both", which="major", labelsize=FONTSIZE*0.7)
    ax.grid(true, alpha=0.5, zorder=1)
    (a-1)%2 == 0 && ax.set_ylabel("Cumulative grounded mass change (Gt)", fontsize=FONTSIZE*0.8)
    (a-1)÷2 == 1 && ax.set_xlabel("Analysis year", fontsize=FONTSIZE*0.8)
end
PyPlot.savefig("../outputs/plots/future_obs/Synthetic_future_trajectories_comparison.pdf", bbox_inches="tight", dpi=300)